# 2단계 모델 검증: 년도 기준 분할 (훈련 ~2023 / 테스트 2024~2025)

## 1. 라이브러리 임포트

In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

## 2. 데이터 불러오기 및 전처리

In [2]:
# 검단 포함 전체 신도시 데이터 불러오기
df = pd.read_csv('real_new_city.csv', encoding='utf-8-sig')

# 거래금액 전처리 (쉼표 제거 후 정수 변환)
df['거래금액(만원)'] = df['거래금액(만원)'].str.replace(',', '').astype(int)

# m2당가격 파생변수 생성
df['m2당가격'] = df['거래금액(만원)'] / df['전용면적(㎡)']

print(f'전체 데이터: {len(df)}건')

전체 데이터: 126375건


## 3. 이상치 제거

In [3]:
# 전용면적 33제곱미터 미만 제거
df = df[df['전용면적(㎡)'] >= 33]

# 발표후경과년수 3년 미만 필터링
before = len(df)
df = df[df['발표후경과년수'] >= 3]
print(f'발표후경과년수 3 미만 제거: {before - len(df)}개 → 남은 데이터: {len(df)}개')

# m2당가격 z-score 기준 이상치 제거 (|z| > 2인 데이터 제거)
mean = df['m2당가격'].mean()
std = df['m2당가격'].std()
z_scores = (df['m2당가격'] - mean) / std
df = df[z_scores.abs() <= 2]

print(f'이상치 제거 후: {len(df)}건')

발표후경과년수 3 미만 제거: 3863개 → 남은 데이터: 122221개
이상치 제거 후: 116935건


## 4. 년도 기준 훈련/테스트 분할

In [4]:
# 독립변수 목록
features = ['건축년도', '층',
            '지하철호선개수', '기차역까지의거리',
            '가장 가까운 지하철역까지의 거리', '가장 가까운 IC와의 거리',
            '발표후경과년수', 'CPI', '계약연도', '서울도심거리',
            '단지별_세대수', '도시별_세대수']

# 결측치 제거
df = df.dropna(subset=features + ['m2당가격'])

# 년도 기준 분할
df_train = df[df['계약연도'] <= 2023]
df_test  = df[df['계약연도'] >= 2024]

# 독립변수와 타겟 분리
train_input  = df_train[features]
train_target = df_train['m2당가격']
test_input   = df_test[features]
test_target  = df_test['m2당가격']

print(f'훈련 데이터 (~2023): {len(df_train)}건')
print(f'테스트 데이터 (2024~2025): {len(df_test)}건')

훈련 데이터 (~2023): 96811건
테스트 데이터 (2024~2025): 20118건


## 5. RandomForest 모델 학습


In [5]:
# RandomForest 모델 학습
rf = RandomForestRegressor(n_jobs=-1, random_state=42)
rf.fit(train_input, train_target)

print(f"{'='*50}")
print(f'2단계 모델 성능 (년도 기준 분할)')
print(f"{'='*50}")
print(f'훈련 R²: {rf.score(train_input, train_target):.4f}')
print(f'테스트 R²: {rf.score(test_input, test_target):.4f}')

2단계 모델 성능 (년도 기준 분할)
훈련 R²: 0.9685
테스트 R²: 0.8021
